In [ ]:
import os
import sys
import pandas as pd
from fetch_audio_features import get_spotify_token, fetch_audio_features_for_csv

# project paths
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))
PREPROCESSING_DIR = os.path.join(PROJECT_ROOT, "src", "preprocessing")
if PREPROCESSING_DIR not in sys.path:
    sys.path.append(PREPROCESSING_DIR)

In [ ]:
# spotify credentials
CLIENT_ID = "6744e5f8277c4936aa125fe90a3cd771"
CLIENT_SECRET = "57df6ceb4b314167a0b6efacee5aa8d3"

def get_token_with_retry(max_retries=3):
    retries = 0
    while retries < max_retries:
        try:
            return get_spotify_token(CLIENT_ID, CLIENT_SECRET)
        except Exception as e:
            retries += 1
            wait = 2 ** retries
            print(f"Error getting Spotify token: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    raise Exception("Failed to get Spotify token after multiple attempts.")

In [ ]:
# paths & years
INTERIM_DIR = os.path.join(PROJECT_ROOT, "data", "interim")
YEARS = [2019, 2020, 2021, 2022, 2023]

# fetch audio features with automatic token refresh
token = get_token_with_retry()

for year in YEARS:
    csv_path = os.path.join(INTERIM_DIR, f"merged_data_{year}.csv")
    if os.path.exists(csv_path):
        success = False
        attempts = 0
        while not success and attempts < 5:
            try:
                fetch_audio_features_for_csv(csv_path, token)
                success = True
            except Exception as e:
                # refresh token if expired or forbidden
                if "expired" in str(e).lower() or "401" in str(e) or "403" in str(e):
                    print(f"Token expired or forbidden. Refreshing token...")
                    token = get_token_with_retry()
                    attempts += 1
                else:
                    print(f"Error processing {csv_path}: {e}")
                    break
    else:
        print(f"CSV not found for year {year}: {csv_path}")